# 1. Scoring a model against its reference

`tcs.compare(model, reference)` returns one row of benchmark metrics for a predicted
TCR-pMHC structure: RMSDs, DockQ, L-RMSDs, lDDT, TM-score and docking-angle errors.

The structures are the package's examples (`tcs.example('3QIB', ...)`, from `tests/data`): 3QIB, a mouse class II complex (the 2B4 TCR on
I-E k), with AF3, Boltz-2 and TCRmodel2 models. Every column is explained in
[docs/metrics.md](../docs/metrics.md).

In [1]:
import pandas as pd

import tcrstruct as tcs
from tcrstruct import TCRFrame

pd.options.display.precision = 3

In [2]:
reference = TCRFrame.read(tcs.example('3QIB', 'rcsb.pdb'), name='3QIB')
reference

TCRFrame '3QIB', MHC class II, chains A, B, C, D, E

In [3]:
print(reference.report())

TCRFrame '3QIB', MHC class II, chains A, B, C, D, E
  A  MHC alpha       H2-EAk       180 aa   (chain A in the file)
  B  MHC beta        H2-EBk       165 aa   (chain B in the file)
  C  peptide                       11 aa   (chain C in the file)
  D  TCR alpha       TRAV4-4/DV10*02 194 aa   (chain D in the file)
  E  TCR beta        TRBV26*01    242 aa   (chain E in the file)
  peptide LIAYLKQATKG   core IAYLKQATK
  CDRs    CDRA1=TTMRA  CDRA2=LASGT  CDRA3=AALRATGGNNKLT  CDRB1=KGHPV  CDRB2=FQNQEV  CDRB3=ASSLNWSQDTQY


## One model

Either argument can be a file, a `TCRFrame` or a plain PDBFrame. Chains are matched by
sequence, so the model's chain IDs and numbering don't matter.

In [4]:
row = tcs.compare(tcs.example('3QIB', 'af3-tbm.cif'), reference, grouped=True)
row

group        column              
pair         PDB                                3QIB
global       Full                              2.562
             lddt                              0.893
             tm_score                          0.982
peptide      Peptide                           1.072
             Peptide_Centric                   1.341
             peptide_reversed                  False
cdrs         CDRA1                             0.952
             CDRA2                             1.141
             CDRA3                             1.415
             CDRB1                             1.949
             CDRB2                             2.811
             CDRB3                             1.302
             Mean_CDR                          1.595
docking      DockQ                             0.771
             iRMSD                              1.03
             LRMSD                             3.483
             fnat                              0.778
            

`grouped=True` gives every column under the group of what it measures (RMSD columns are
named by their region: `Full`, `Peptide`, `CDRA1`, ...):

| group | columns |
|---|---|
| `'global'` | `Full`, `lddt`, `tm_score` |
| `'peptide'` | `Peptide`, `Peptide_Centric`, `peptide_reversed` |
| `'cdrs'` | `CDRA1` ... `CDRB3`, `Mean_CDR` |
| `'docking'` | `DockQ`, `iRMSD`, `LRMSD`, `TCR_LRMSD`, `pMHC_LRMSD`, `fnat`, `F1` |
| `'angles'` | the model's `crossing_angle`, `incident_angle`, and their errors |
| `'contacts'`, `'bookkeeping'` | DockQ's contact counts; residue counts, identity, the superposition |

So one group is `row['docking']`:

In [5]:
row['docking']

column
DockQ         0.771
iRMSD          1.03
LRMSD         3.483
fnat          0.778
F1            0.824
TCR_LRMSD     3.105
pMHC_LRMSD    1.047
Name: af3-tbm, dtype: object

## Several models

`tcs.compare_models` scores several models of one reference: one row per model, as
numbers.

In [6]:
models = {'AF3': tcs.example('3QIB', 'af3-tbm.cif'),
          'Boltz-2': tcs.example('3QIB', 'boltz-2.pdb'),
          'TCRmodel2': tcs.example('3QIB', 'tcrmodel2.pdb')}
table = tcs.compare_models(models, reference, grouped=True)
table[['global', 'peptide', 'cdrs']]

group     global                 peptide                                   \
column      Full   lddt tm_score Peptide Peptide_Centric peptide_reversed   
model                                                                       
AF3        2.562  0.893    0.982   1.072           1.341            False   
Boltz-2    1.084  0.945    0.994   0.250           0.614            False   
TCRmodel2  1.781  0.907    0.991   0.746           1.058            False   

group       cdrs                                              
column     CDRA1  CDRA2  CDRA3  CDRB1  CDRB2  CDRB3 Mean_CDR  
model                                                         
AF3        0.952  1.141  1.415  1.949  2.811  1.302    1.595  
Boltz-2    0.288  0.292  0.316  0.711  1.179  0.378    0.527  
TCRmodel2  1.067  0.918  1.276  0.905  1.243  0.998    1.068

In [7]:
table[['docking', 'angles']]

group     docking                                                   \
column      DockQ  iRMSD  LRMSD   fnat     F1 TCR_LRMSD pMHC_LRMSD   
model                                                                
AF3         0.771  1.030  3.483  0.778  0.824     3.105      1.047   
Boltz-2     0.961  0.260  1.389  0.939  0.964     1.111      0.486   
TCRmodel2   0.904  0.529  1.099  0.838  0.888     2.232      0.269   

group             angles                                      \
column    crossing_angle incident_angle crossing_angle_error   
model                                                          
AF3               41.311         15.474                4.394   
Boltz-2           38.855         11.775                1.938   
TCRmodel2         38.584         10.250                1.667   

group                           
column    incident_angle_error  
model                           
AF3                      5.057  
Boltz-2                  1.358  
TCRmodel2                0.245

## Seeing the differences

`view_comparison` shows the models on the reference, each colored by its per-residue
lDDT, from red (low) to blue (high). Here the fit uses only the pMHC (chains A, B, C), so
any difference in how the TCR docks shows up as an offset. `regions=` takes any region or
group in `tcs.list_regions()` and `tcs.list_region_groups()`.

In [8]:
tcs.view_comparison(models, reference, superpose=['A', 'B', 'C'], regions='cdr3')

## Which residues are off

`compare` gives one lDDT for the whole model. `tcs.per_residue_lddt` gives it per residue
(with its IMGT label and region), or as the mean per region, worst first:

In [9]:
tcs.per_residue_lddt(models['AF3'], reference, by='region')

region
CDRB2           0.752
peptide         0.788
CDRA3           0.825
CDRB3           0.833
CDRB1           0.846
peptide_core    0.856
CDRA1           0.881
helices         0.885
CDRA2           0.891
Vb              0.892
Va              0.914
Name: lddt, dtype: float64

`compare` runs the steps in `tcs.metrics` one after another: `match` (the atoms both
structures share), `superpose`, then `rmsds`, `dockq`, `lddt`, `tm_score`, `angles`. Use
`tcs.metrics` only for something `compare` and `per_residue_lddt` don't give.

## Options

- `align_on=`: what the superposition fits on. The default is `'peptide_centric'`;
  others are `'full'`, `'pmhc'`, `'mhc'`, `'tcr'`, `'v_domains'`, `'cdrs'`. A list scores
  each of them, one row per superposition.
- `align_atoms=`: fit the Cα atoms of that region (`'ca'`, default) or all its heavy
  atoms (`'heavy'`).
- `**tcs.V1`: the `'v1'` versions of DockQ, TM-score and the angles,
  with a heavy-atom fit.

With `align_on='pmhc'`, `TCR_LRMSD` is the pMHC-aligned L-RMSD: how far the TCR is from
where it should sit on the pMHC.

In [10]:
fits = tcs.compare(models['AF3'], reference, align_on=['peptide_centric', 'full', 'pmhc'],
                   grouped=True)
fits[['global', 'docking']]

group           global                 docking                             \
column            Full   lddt tm_score   DockQ iRMSD  LRMSD   fnat     F1   
peptide_centric  2.562  0.893    0.982   0.771  1.03  3.483  0.778  0.824   
full             1.615  0.893    0.982   0.771  1.03  3.483  0.778  0.824   
pmhc             2.652  0.893    0.982   0.771  1.03  3.483  0.778  0.824   

group                                 
column          TCR_LRMSD pMHC_LRMSD  
peptide_centric     3.105      1.047  
full                1.366      1.286  
pmhc                3.313      0.622

The RMSDs change with the superposition; lDDT, the angles and the default (`'v2'`) DockQ
and TM-score don't, so they are computed once. The `'v1'` versions:

In [11]:
v1 = tcs.compare(models['AF3'], reference, grouped=True, **tcs.V1)
pd.DataFrame({'default': row, 'tcs.V1': v1}).loc[['global', 'cdrs', 'docking', 'angles']]

default  tcs.V1
group   column                              
global  Full                   2.562   2.459
        lddt                   0.893   0.893
        tm_score               0.982    0.95
cdrs    CDRA1                  0.952   0.947
        CDRA2                  1.141   1.082
        CDRA3                  1.415   1.409
        CDRB1                  1.949   1.979
        CDRB2                  2.811   2.809
        CDRB3                  1.302   1.313
        Mean_CDR               1.595    1.59
docking DockQ                  0.771    0.89
        iRMSD                   1.03   0.833
        LRMSD                  3.483   0.692
        fnat                   0.778   0.912
        F1                     0.824   0.912
        TCR_LRMSD              3.105   2.954
        pMHC_LRMSD             1.047   1.025
angles  crossing_angle        41.311  35.544
        incident_angle        15.474  36.219
        crossing_angle_error   4.394   4.292
        incident_angle_error   5.057   8.906

The package versions are kept with the result, not in a column:

In [12]:
row.attrs['versions']

'tcrstruct 0.3.20, pdbframe 0.1.5, tcrkit 0.1.5'

To score whole folders of models, use `tcs.compare_dirs(models_dir, references_dir)`, or
`tcrstruct compare-dirs` on the command line, which also writes the settings and versions
to `<output>.settings.json`. It pairs files by PDB ID and returns one row per model.